In [ ]:
import ssl
import certifi
import urllib.request

# Patch SSL to use certifi's certificates instead of Windows store
ssl._create_default_https_context = ssl.create_default_context
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())

import ee
import geemap
import geopandas as gpd

# ee.Authenticate(force= True)


Successfully saved authorization token.


In [ ]:
import ee
import geemap
import geopandas as gpd
import pandas as pd
import os

# 1. AUTHENTICATE WITH YOUR NEW PROJECT ID HERE
ee.Initialize(project='abve-499717')

print("Booting up the coordinate generator...")

# 2. Load your local district boundaries
gdf = gpd.read_file('../../data/INDIA_DISTRICTS.geojson')
agri_belt_gdf = gdf[gdf['state'].str.upper().isin(['PUNJAB', 'HARYANA'])]
ee_agri_belt = geemap.geopandas_to_ee(agri_belt_gdf)

# 3. Pull the ESA Crop Mask (Class 40 = Cropland)
world_cover = ee.ImageCollection("ESA/WorldCover/v200").first()
crop_mask = world_cover.eq(40).clip(ee_agri_belt)

# 4. Drop 1000 random pins and filter out the cities/lakes
print("Dropping 1,000 pins on active farmlands...")
random_points = ee.FeatureCollection.randomPoints(ee_agri_belt.geometry(), 1000, seed=42)

farm_points = crop_mask.reduceRegions(
    collection=random_points, 
    reducer=ee.Reducer.first(), 
    scale=10
).filter(ee.Filter.eq('first', 1))

# 5. Extract the Lat/Lon data locally
print("Extracting GPS coordinates...")
coords = farm_points.getInfo()['features']

# 6. Format into a Pandas DataFrame and save
data = []
for i, feature in enumerate(coords):
    lon, lat = feature['geometry']['coordinates']
    data.append({'point_id': i+1, 'lat': lat, 'lon': lon})

df = pd.DataFrame(data)

# Create the data folder if it doesn't exist
os.makedirs('data', exist_ok=True)

# Save the final CSV
df.to_csv('../../data/sample_points.csv', index=False)
print("Boom. sample_points.csv generated successfully. You can now run the main extraction script.")

Booting up the coordinate generator...
Dropping 1,000 pins on active farmlands...
Extracting GPS coordinates...
Boom. sample_points.csv generated successfully. You can now run the main extraction script.
